# Translation  Cross-Validation

# Step 1: Input & clean sequences

In [1]:
dna_fs = input("Enter the fusion sequence: ")
gene_a_cds = input("Enter Gene A CDS sequence: ")
gene_b_cds = input("Enter Gene B CDS sequence: ")

gene_a = gene_a_cds.replace(" ", "").upper()
gene_b = gene_b_cds.replace(" ", "").upper()
dna_fs = dna_fs.replace(" ", "").upper()


Enter the fusion sequence:  CGCCAAGTGGCCGTACCGCGGCGGCGGCGGCGGCGGCAGCGCGGGGGGCG*GCGGGCAGTTCGGCGGCGCGGGGCCCGGGGCCGGGGGCGGCGGCGGCCCC
Enter Gene A CDS sequence:  atg aacttccagg cgggcggggg gcagagcccg cagcagcagc agagcctggc ggctccgggg ggcggcggcg ctgccgcgca gcagctcgtc tgcggcgggc agttcggcgg cgcggggccc ggggccgggg gcggcggcgg cccctcgcag cagctggccg gcgggccccc ccagcagttc gcgctctcca actccgcggc catccgggcc gagatccagc gcttcgagtc cgtgcatccc aatatctacg ccatctacga cctgatcgag cgcatcgagg atttggcgct gcagaaccag atccgggagc acgtcatctc catcgaggac tcgtttgtga acagccagga gtggacgctg agccgctccg taccggagct taaagtgggc atagtgggga acctgtctag cgggaagtca gccctggtgc accgctatct gacggggacc tatgtccagg aggagtcccc tgaagggggg cggtttaaga aggagattgt ggtggatggc cagagttacc tgctgctgat ccgagatgaa ggaggccccc ctgagctcca gtttgctgcc tgggtggatg cagtggtgtt tgtgttcagc ctggaggatg aaatcagttt ccagacggtg tacaactact tcctgcgtct ctgcagcttc cgcaacgcca gcgaggtgcc catggtgctt gtgggcacgc aggatgccat cagcgctgcg aatccccggg ttatcgacga cagcagagcc cgcaagctct cc

# Step 2: Split te fusion seq and search each fragment seperately

In [2]:
left_fragment, right_fragment = dna_fs.split("*")

def search(fragment, gene):

    position = gene.find(fragment)

    if position == -1:
        return "Not found"

    return position


print("\nLEFT FRAGMENT")
print("Gene A:", search(left_fragment, gene_a))
print("Gene B:", search(left_fragment, gene_b))

print("\nRIGHT FRAGMENT")
print("Gene A:", search(right_fragment, gene_a))
print("Gene B:", search(right_fragment, gene_b))


left_in_a = gene_a.find(left_fragment)
left_in_b = gene_b.find(left_fragment)

right_in_a = gene_a.find(right_fragment)
right_in_b = gene_b.find(right_fragment)




LEFT FRAGMENT
Gene A: Not found
Gene B: 209

RIGHT FRAGMENT
Gene A: 97
Gene B: Not found


# Step 3: Reconstruct fusion CDS

In [3]:
# We also record, generically, which gene sits UPSTREAM (5') and which sits DOWNSTREAM (3') of the junction, and the downstream 
# gene's own native index at the breakpoint. This generic "downstream_index" is the number we need later to test whether
# the fusion preserves or disrupts the downstream gene's own reading frame.

if left_in_a != -1 and right_in_b != -1:

    left_sequence = gene_a[:left_in_a + len(left_fragment)]
    right_sequence = gene_b[right_in_b:]

    upstream_gene_name = "Gene A"
    downstream_gene_name = "Gene B"
    downstream_index = right_in_b

elif left_in_b != -1 and right_in_a != -1:

    left_sequence = gene_b[:left_in_b + len(left_fragment)]
    right_sequence = gene_a[right_in_a:]

    upstream_gene_name = "Gene B"
    downstream_gene_name = "Gene A"
    downstream_index = right_in_a

else:

    print("\nUnable to reconstruct the fusion CDS.")
    exit()


fusion_cds = left_sequence + right_sequence

print("\nFusion CDS:\n")
print(fusion_cds)

print("\nUpstream gene (5' of junction):", upstream_gene_name)
print("Downstream gene (3' of junction):", downstream_gene_name)



Fusion CDS:

ATGAACGACTTTGACGAGTGCGGCCAGAGCGCAGCCAGCATGTACCTGCCGGGCTGCGCCTACTATGTGGCCCCGTCTGACTTCGCTAGCAAGCCTTCGTTCCTTTCCCAACCGTCGTCCTGCCAGATGACTTTCCCCTACTCTTCCAACCTGGCTCCGCACGTCCAGCCCGTGCGCGAAGTGGCCTTCCGCGACTACGGCCTGGAGCGCGCCAAGTGGCCGTACCGCGGCGGCGGCGGCGGCGGCAGCGCGGGGGGCGGCGGGCAGTTCGGCGGCGCGGGGCCCGGGGCCGGGGGCGGCGGCGGCCCCTCGCAGCAGCTGGCCGGCGGGCCCCCCCAGCAGTTCGCGCTCTCCAACTCCGCGGCCATCCGGGCCGAGATCCAGCGCTTCGAGTCCGTGCATCCCAATATCTACGCCATCTACGACCTGATCGAGCGCATCGAGGATTTGGCGCTGCAGAACCAGATCCGGGAGCACGTCATCTCCATCGAGGACTCGTTTGTGAACAGCCAGGAGTGGACGCTGAGCCGCTCCGTACCGGAGCTTAAAGTGGGCATAGTGGGGAACCTGTCTAGCGGGAAGTCAGCCCTGGTGCACCGCTATCTGACGGGGACCTATGTCCAGGAGGAGTCCCCTGAAGGGGGGCGGTTTAAGAAGGAGATTGTGGTGGATGGCCAGAGTTACCTGCTGCTGATCCGAGATGAAGGAGGCCCCCCTGAGCTCCAGTTTGCTGCCTGGGTGGATGCAGTGGTGTTTGTGTTCAGCCTGGAGGATGAAATCAGTTTCCAGACGGTGTACAACTACTTCCTGCGTCTCTGCAGCTTCCGCAACGCCAGCGAGGTGCCCATGGTGCTTGTGGGCACGCAGGATGCCATCAGCGCTGCGAATCCCCGGGTTATCGACGACAGCAGAGCCCGCAAGCTCTCCACAGATCTGAAGCGGTGCACCTACTATGAGACGTGCGCGACCTACGGGCTCAATGTGGA

# Step 4: Save Junction position

In [4]:
junction = len(left_sequence)

print("\nDNA junction position:", junction)
print("Reading frame (junction mod 3):", junction % 3)


DNA junction position: 259
Reading frame (junction mod 3): 1


# Step 5: Translate fusion CDS

In [5]:
# Two changes from the basic version:
#
# 1. Translation now STOPS at the first stop codon, matching real ribosome behaviour. This matters a lot for out-of-frame
#    fusions, which can hit an unpredictable premature stop.
#
# 2. We explicitly determine whether the downstream gene is being read in its native frame, or a shifted frame, using:
#
#       junction % 3 == downstream_index % 3   ->  frame PRESERVED
#       junction % 3 != downstream_index % 3   ->  frame SHIFTED
#
#    This is the general test your supervisor asked for. It works for in-frame AND out-of-frame fusions alike.

genetic_code = {

    "ATA":"I","ATC":"I","ATT":"I","ATG":"M",
    "ACA":"T","ACC":"T","ACG":"T","ACT":"T",
    "AAC":"N","AAT":"N","AAA":"K","AAG":"K",
    "AGC":"S","AGT":"S","AGA":"R","AGG":"R",
    "CTA":"L","CTC":"L","CTG":"L","CTT":"L",
    "CCA":"P","CCC":"P","CCG":"P","CCT":"P",
    "CAC":"H","CAT":"H","CAA":"Q","CAG":"Q",
    "CGA":"R","CGC":"R","CGG":"R","CGT":"R",
    "GTA":"V","GTC":"V","GTG":"V","GTT":"V",
    "GCA":"A","GCC":"A","GCG":"A","GCT":"A",
    "GAC":"D","GAT":"D","GAA":"E","GAG":"E",
    "GGA":"G","GGC":"G","GGG":"G","GGT":"G",
    "TCA":"S","TCC":"S","TCG":"S","TCT":"S",
    "TTC":"F","TTT":"F","TTA":"L","TTG":"L",
    "TAC":"Y","TAT":"Y","TAA":"*","TAG":"*","TGA":"*"

}

protein = ""
stop_codon_hit = False

for i in range(0, len(fusion_cds) - 2, 3):

    codon = fusion_cds[i:i + 3]

    amino_acid = genetic_code.get(codon, "X")

    if amino_acid == "*":
        stop_codon_hit = True
        break

    protein += amino_acid


junction_aa = junction // 3

print("\nFusion protein sequence (stops at first stop codon):\n")
print(protein)

if stop_codon_hit:
    print("\nTranslation halted at a stop codon.")
else:
    print("\nNo stop codon encountered before the sequence ran out.")

print("\nProtein junction position:", junction_aa)

if junction_aa < len(protein):
    print("Amino acid at junction:", protein[junction_aa])
else:
    print("Junction falls beyond the translated protein "
          "(likely due to an early stop codon).")


# --- Frame status: does downstream match the downstream gene's own frame? ---

if junction % 3 == downstream_index % 3:
    frame_status = "preserved"
else:
    frame_status = "shifted"

print("\nDownstream reading frame relative to", downstream_gene_name + ":", frame_status)

if frame_status == "preserved":
    print("Downstream amino acids match", downstream_gene_name,
          "'s native protein. Novelty is confined to peptides")
    print("that literally overlap the junction residue.")
else:
    print("Downstream amino acids are shifted out of",
          downstream_gene_name, "'s native frame.")
    print("Everything from the junction to the stop codon is novel sequence.")



Fusion protein sequence (stops at first stop codon):

MNDFDEXGQSAASMYLPGXAYYVAPSDFASKPSFLSQPSSXQMTFPYSSNLAPHVQPVREVAFRDYGLERAKXPYRGGGGGGSAGGGGQFGGAGPGAGGGGGPSQQLAGGPPQQFALSNSAAIRAEIQRFESVHPNIYAIYDLIERIEDLALQNQIREHVISIEDSFVNSQEXTLSRSVPELKVGIVGNLSSGKSALVHRYLTGTYVQEESPEGGRFKKEIVVDGQSYLLLIRDEGGPPELQFAAXVDAVVFVFSLEDEISFQTVYNYFLRLXSFRNASEVPMVLVGTQDAISAANPRVIDDSRARKLSTDLKRXTYYETXATYGLNVERVFQDVAQKVVALRKKQQLAIGPXKSLPNSPSHSAVSAASIPAVHINQATNGGGSAFSDYSSSVPSTPSISQRELRIETIAASSTPTPIRKQSKRRSNIFTSRKGADLDREKKAAEXKVDSIGSGRAIPIKQGILLKRSGKSLNKEXKKKYVTLXDNGLLTYHPSLHDYMQNIHGKEIDLLRTTVKVPGKRLPRATPATAPGTSPRANGLSVERSNTQLGGGTGAPHSASSASLHSERPLSSSAXAGPRPEGLHQRSXSVSSADQXSEATTSLPPGMQHPASGPAEVLSSSPKLDPPPSPHSNRKKHRRKKSTGTPRPDGPSSATEEAEESFEFVVVSLTGQTXHFEASTAEERELXVQSVQAQILASLQGXRSAKDKTRLGNQNAALAVQAVRTVRGNSFXIDXDAPNPDXASLNLGALMXIEXSGIHRHLGAHLSRVRSLDLDDXPPELLAVMTAMGNALANSVXEGALGGYSKPGPDAXREEKERXIRAKYEQKLFLAPLPSSDVPLGQQLLRAVVEDDLRLLVMLLAHGSKEEVNETYGDGDGRTALHLSSAMANVVFTQLLIXYGVDVRSRDARGLTPLAYARRAGSQEXADILIQHGXPGEGXGLAPTPN

# Step 6: Define the novel region (in amino acid coordinates)

In [6]:
# preserved -> only the single junction residue is "new context"
# shifted   -> junction_aa through the end of the translated
#              protein (i.e. up to wherever translation stopped)
#              is entirely novel

if frame_status == "preserved":
    novel_start_aa = junction_aa
    novel_end_aa = junction_aa            # inclusive
else:
    novel_start_aa = junction_aa
    novel_end_aa = len(protein) - 1       # inclusive, protein already stops at first stop codon

print("\nNovel region (amino acid coordinates):",
      novel_start_aa, "to", novel_end_aa)


Novel region (amino acid coordinates): 86 to 86


# Step 7: Generate Junction / Novel spanning peptides

In [7]:
window_size = int(input("\nEnter peptide length: "))


Enter peptide length:  8


In [8]:
# METHOD A: brute-force filter over every window 
# Generate every possible peptide window in the whole protein, keep only the ones overlapping the novel region.

all_peptides = []

for i in range(len(protein) - window_size + 1):

    peptide = protein[i:i + window_size]

    all_peptides.append((i, peptide))


method_a_peptides = []

for position, peptide in all_peptides:

    window_start = position
    window_end = position + window_size - 1   # inclusive

    overlaps_novel_region = (
        window_start <= novel_end_aa and window_end >= novel_start_aa
    )

    if overlaps_novel_region:
        method_a_peptides.append((position, peptide))

In [11]:
window_size = int(input("\nEnter peptide length: "))


Enter peptide length:  8


In [12]:
# ---------- METHOD B: direct computed start range ----------
# Instead of generating every window in the whole protein, work out mathematically which start positions COULD overlap the
# novel region, then slice only those directly.

earliest_start = max(0, novel_start_aa - window_size + 1)
latest_start = min(len(protein) - window_size, novel_end_aa)

method_b_peptides = []

for position in range(earliest_start, latest_start + 1):

    peptide = protein[position:position + window_size]

    method_b_peptides.append((position, peptide))



In [13]:
window_size = int(input("\nEnter peptide length: "))


Enter peptide length:  8


In [14]:

# ---------- METHOD C: offset-from-junction reconstruction ----------
# Re-derive the same peptides from a completely different angle: walk outward from the junction residue itself using an "offset"
# (how many residues before the junction are included), rather than scanning start positions across the whole protein.

method_c_peptides = []
seen_positions = set()

for offset in range(window_size):

    position = novel_start_aa - offset

    if position < 0 or position + window_size > len(protein):
        continue

    peptide = protein[position:position + window_size]

    if position not in seen_positions:
        method_c_peptides.append((position, peptide))
        seen_positions.add(position)

if frame_status == "shifted":

    # Also add windows that sit fully downstream of the junction, 
    # i.e. they no longer contain novel_start_aa itself but are still entirely inside the novel (shifted-frame) region.

    for position in range(novel_start_aa + 1, novel_end_aa + 1):

        if position + window_size > len(protein):
            continue

        if position in seen_positions:
            continue

        peptide = protein[position:position + window_size]
        method_c_peptides.append((position, peptide))
        seen_positions.add(position)

method_c_peptides.sort(key=lambda item: item[0])



# CROSS-CHECK ALL THREE METHODs

In [15]:
set_a = set(method_a_peptides)
set_b = set(method_b_peptides)
set_c = set(method_c_peptides)

print("\n" + "=" * 60)
print("JUNCTION / NOVEL-SPANNING PEPTIDES")
print("=" * 60)

print("\nMethod A (brute-force filter):")
for position, peptide in method_a_peptides:
    print(position, peptide)

print("\nMethod B (direct computed range):")
for position, peptide in method_b_peptides:
    print(position, peptide)

print("\nMethod C (offset-from-junction):")
for position, peptide in method_c_peptides:
    print(position, peptide)

print("\n" + "=" * 60)

if set_a == set_b == set_c:
    print("All three methods agree exactly:", len(set_a), "peptide(s) found.")
else:
    print("WARNING: the three methods do NOT agree. Investigate before trusting output.")
    print("Method A unique to it:", set_a - (set_b | set_c))
    print("Method B unique to it:", set_b - (set_a | set_c))
    print("Method C unique to it:", set_c - (set_a | set_b))

# fusion_peptides kept for compatibility with earlier naming
fusion_peptides = method_a_peptides


JUNCTION / NOVEL-SPANNING PEPTIDES

Method A (brute-force filter):
79 GGGSAGGG
80 GGSAGGGG
81 GSAGGGGQ
82 SAGGGGQF
83 AGGGGQFG
84 GGGGQFGG
85 GGGQFGGA
86 GGQFGGAG

Method B (direct computed range):
79 GGGSAGGG
80 GGSAGGGG
81 GSAGGGGQ
82 SAGGGGQF
83 AGGGGQFG
84 GGGGQFGG
85 GGGQFGGA
86 GGQFGGAG

Method C (offset-from-junction):
79 GGGSAGGG
80 GGSAGGGG
81 GSAGGGGQ
82 SAGGGGQF
83 AGGGGQFG
84 GGGGQFGG
85 GGGQFGGA
86 GGQFGGAG

All three methods agree exactly: 8 peptide(s) found.
